# Silver Layer EDA: Vitals Data Quality Analysis

This notebook performs exploratory data analysis on `silver_vitals.parquet` to:
- Identify data quality issues
- Detect outliers
- Inform the cleaning strategy for the gold layer

## Cell 1: Setup

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SILVER_PATH = Path("../output/silver/silver_vitals.parquet")
conn = duckdb.connect(":memory:")

# Load data into DuckDB
conn.execute(f"CREATE VIEW silver_vitals AS SELECT * FROM read_parquet('{SILVER_PATH}')")
print(f"Data loaded from {SILVER_PATH}")

## Cell 2: Data Overview

In [ ]:
# Row count and column info
print("=== Data Overview ===")
row_count = conn.execute("SELECT COUNT(*) FROM silver_vitals").fetchone()[0]
print(f"Total rows: {row_count:,}")

print("\n=== Column Types ===")
schema = conn.execute("DESCRIBE silver_vitals").df()
display(schema)

print("\n=== Sample Rows ===")
sample = conn.execute("SELECT * FROM silver_vitals USING SAMPLE 5").df()
display(sample)

print("\n=== LOINC Code Distribution ===")
loinc_dist = conn.execute("""
    SELECT 
        loinc_code,
        COUNT(*) as count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as pct
    FROM silver_vitals
    GROUP BY loinc_code
    ORDER BY count DESC
""").df()
display(loinc_dist)

print("\n=== Unique Counts ===")
unique_counts = conn.execute("""
    SELECT 
        COUNT(DISTINCT patient_id) as unique_patients,
        COUNT(DISTINCT encounter_id) as unique_encounters
    FROM silver_vitals
""").df()
display(unique_counts)

## Cell 3: Physiological Reference Limits

In [ ]:
LIMITS = {
    "8310-5": {"name": "body_temperature", "unit": "°F", "min": 90, "max": 110},
    "8867-4": {"name": "heart_rate", "unit": "bpm", "min": 20, "max": 300},
    "9279-1": {"name": "respiratory_rate", "unit": "/min", "min": 4, "max": 60},
    "2708-6": {"name": "oxygen_saturation", "unit": "%", "min": 50, "max": 100},
    "8480-6": {"name": "systolic_bp", "unit": "mmHg", "min": 40, "max": 300},
    "8462-4": {"name": "diastolic_bp", "unit": "mmHg", "min": 20, "max": 200},
}

# Display limits as a table
limits_df = pd.DataFrame([
    {"loinc_code": k, **v} for k, v in LIMITS.items()
])
print("=== Physiological Reference Limits ===")
display(limits_df)

## Cell 4: Percentile Analysis

In [ ]:
print("=== Percentile Analysis by Vital ===")

percentile_query = """
SELECT
    loinc_code,
    COUNT(*) as n,
    ROUND(MIN(value), 2) as min_val,
    ROUND(PERCENTILE_CONT(0.001) WITHIN GROUP (ORDER BY value), 2) as p0_1,
    ROUND(PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY value), 2) as p01,
    ROUND(PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY value), 2) as p05,
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY value), 2) as p25,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY value), 2) as p50,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY value), 2) as p75,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY value), 2) as p95,
    ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY value), 2) as p99,
    ROUND(PERCENTILE_CONT(0.999) WITHIN GROUP (ORDER BY value), 2) as p99_9,
    ROUND(MAX(value), 2) as max_val
FROM silver_vitals
GROUP BY loinc_code
ORDER BY loinc_code
"""

percentiles_df = conn.execute(percentile_query).df()

# Add vital names for readability
percentiles_df['vital_name'] = percentiles_df['loinc_code'].map(lambda x: LIMITS.get(x, {}).get('name', 'unknown'))
cols = ['loinc_code', 'vital_name'] + [c for c in percentiles_df.columns if c not in ['loinc_code', 'vital_name']]
percentiles_df = percentiles_df[cols]

display(percentiles_df)

# Count values outside physiological limits
print("\n=== Values Outside Physiological Limits ===")

outlier_counts = []
for loinc, limits in LIMITS.items():
    result = conn.execute(f"""
        SELECT
            '{loinc}' as loinc_code,
            '{limits['name']}' as vital_name,
            COUNT(*) as total,
            SUM(CASE WHEN value < {limits['min']} THEN 1 ELSE 0 END) as below_min,
            SUM(CASE WHEN value > {limits['max']} THEN 1 ELSE 0 END) as above_max,
            ROUND(100.0 * SUM(CASE WHEN value < {limits['min']} THEN 1 ELSE 0 END) / COUNT(*), 4) as pct_below,
            ROUND(100.0 * SUM(CASE WHEN value > {limits['max']} THEN 1 ELSE 0 END) / COUNT(*), 4) as pct_above
        FROM silver_vitals
        WHERE loinc_code = '{loinc}'
    """).df()
    outlier_counts.append(result)

outliers_df = pd.concat(outlier_counts, ignore_index=True)
display(outliers_df)

## Cell 5: Distribution Histograms

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (loinc, limits) in enumerate(LIMITS.items()):
    ax = axes[idx]
    
    # Get data for this vital (clipped for visualization)
    clip_min = limits['min'] - (limits['max'] - limits['min']) * 0.1
    clip_max = limits['max'] + (limits['max'] - limits['min']) * 0.1
    
    data = conn.execute(f"""
        SELECT value FROM silver_vitals 
        WHERE loinc_code = '{loinc}'
        AND value BETWEEN {clip_min} AND {clip_max}
    """).df()['value']
    
    ax.hist(data, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(limits['min'], color='red', linestyle='--', label=f"Min: {limits['min']}")
    ax.axvline(limits['max'], color='red', linestyle='--', label=f"Max: {limits['max']}")
    ax.set_title(f"{limits['name']} ({loinc})")
    ax.set_xlabel(f"Value ({limits['unit']})")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle("Vital Sign Distributions with Physiological Limits", y=1.02, fontsize=14)
plt.show()

# Log-scale version for extreme outliers
print("\n=== Log-Scale Histograms (Full Range) ===")
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (loinc, limits) in enumerate(LIMITS.items()):
    ax = axes[idx]
    
    data = conn.execute(f"""
        SELECT value FROM silver_vitals 
        WHERE loinc_code = '{loinc}'
    """).df()['value']
    
    ax.hist(data, bins=100, edgecolor='black', alpha=0.7, log=True)
    ax.axvline(limits['min'], color='red', linestyle='--', label=f"Min: {limits['min']}")
    ax.axvline(limits['max'], color='red', linestyle='--', label=f"Max: {limits['max']}")
    ax.set_title(f"{limits['name']} ({loinc}) - Log Scale")
    ax.set_xlabel(f"Value ({limits['unit']})")
    ax.set_ylabel("Log Frequency")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Cell 6: Outlier Deep Dive

In [ ]:
print("=== Outlier Deep Dive ===")

for loinc, limits in LIMITS.items():
    print(f"\n--- {limits['name']} ({loinc}) ---")
    print(f"Expected range: {limits['min']} - {limits['max']} {limits['unit']}")
    
    # Below minimum
    below_min = conn.execute(f"""
        SELECT value, COUNT(*) as count
        FROM silver_vitals
        WHERE loinc_code = '{loinc}' AND value < {limits['min']}
        GROUP BY value
        ORDER BY count DESC
        LIMIT 10
    """).df()
    
    if len(below_min) > 0:
        print(f"\nTop 10 values below minimum ({limits['min']}):")
        display(below_min)
    else:
        print(f"\nNo values below minimum ({limits['min']})")
    
    # Above maximum
    above_max = conn.execute(f"""
        SELECT value, COUNT(*) as count
        FROM silver_vitals
        WHERE loinc_code = '{loinc}' AND value > {limits['max']}
        GROUP BY value
        ORDER BY count DESC
        LIMIT 10
    """).df()
    
    if len(above_max) > 0:
        print(f"\nTop 10 values above maximum ({limits['max']}):")
        display(above_max)
    else:
        print(f"\nNo values above maximum ({limits['max']})")

## Cell 7: Null and Zero Analysis

In [ ]:
print("=== Null and Zero Analysis ===")

null_zero_query = """
SELECT
    loinc_code,
    COUNT(*) as total,
    SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) as null_count,
    ROUND(100.0 * SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) as null_pct,
    SUM(CASE WHEN value = 0 THEN 1 ELSE 0 END) as zero_count,
    ROUND(100.0 * SUM(CASE WHEN value = 0 THEN 1 ELSE 0 END) / COUNT(*), 4) as zero_pct
FROM silver_vitals
GROUP BY loinc_code
ORDER BY loinc_code
"""

null_zero_df = conn.execute(null_zero_query).df()
null_zero_df['vital_name'] = null_zero_df['loinc_code'].map(lambda x: LIMITS.get(x, {}).get('name', 'unknown'))
cols = ['loinc_code', 'vital_name'] + [c for c in null_zero_df.columns if c not in ['loinc_code', 'vital_name']]
null_zero_df = null_zero_df[cols]

display(null_zero_df)

# Check if zeros are sentinel values
print("\n=== Zero Value Investigation ===")
print("Zeros are likely invalid for: body_temperature, heart_rate, respiratory_rate, systolic_bp, diastolic_bp")
print("Zeros might be valid for: oxygen_saturation (though clinically unlikely)")

## Cell 8: Temporal Analysis

In [ ]:
print("=== Temporal Analysis ===")

# Measurements per encounter
print("\n--- Measurements per Encounter ---")
measurements_per_encounter = conn.execute("""
    SELECT
        PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY cnt) as p01,
        PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY cnt) as p05,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY cnt) as p25,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY cnt) as p50,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY cnt) as p75,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY cnt) as p95,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY cnt) as p99,
        MIN(cnt) as min_cnt,
        MAX(cnt) as max_cnt
    FROM (
        SELECT encounter_id, COUNT(*) as cnt
        FROM silver_vitals
        GROUP BY encounter_id
    )
""").df()
display(measurements_per_encounter)

# Distribution histogram
enc_counts = conn.execute("""
    SELECT cnt, COUNT(*) as num_encounters
    FROM (
        SELECT encounter_id, COUNT(*) as cnt
        FROM silver_vitals
        GROUP BY encounter_id
    )
    GROUP BY cnt
    ORDER BY cnt
""").df()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(enc_counts['cnt'], enc_counts['num_encounters'], width=1, edgecolor='black', alpha=0.7)
ax.set_xlabel("Number of Measurements per Encounter")
ax.set_ylabel("Number of Encounters")
ax.set_title("Distribution of Measurements per Encounter")
ax.set_yscale('log')
plt.show()

# Encounters with very few measurements
print("\n--- Encounters with Very Few Measurements ---")
sparse_encounters = conn.execute("""
    SELECT cnt as measurements, COUNT(*) as num_encounters
    FROM (
        SELECT encounter_id, COUNT(*) as cnt
        FROM silver_vitals
        GROUP BY encounter_id
    )
    WHERE cnt <= 6
    GROUP BY cnt
    ORDER BY cnt
""").df()
display(sparse_encounters)

# Time range of data
print("\n--- Time Range ---")
time_range = conn.execute("""
    SELECT 
        MIN(effective_datetime) as earliest,
        MAX(effective_datetime) as latest
    FROM silver_vitals
""").df()
display(time_range)

## Cell 9: Unit Consistency Check

In [ ]:
print("=== Unit Consistency Check ===")

unit_dist = conn.execute("""
    SELECT 
        loinc_code,
        unit,
        COUNT(*) as count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY loinc_code), 2) as pct_of_vital
    FROM silver_vitals
    GROUP BY loinc_code, unit
    ORDER BY loinc_code, count DESC
""").df()

unit_dist['vital_name'] = unit_dist['loinc_code'].map(lambda x: LIMITS.get(x, {}).get('name', 'unknown'))
unit_dist['expected_unit'] = unit_dist['loinc_code'].map(lambda x: LIMITS.get(x, {}).get('unit', 'unknown'))
cols = ['loinc_code', 'vital_name', 'unit', 'expected_unit', 'count', 'pct_of_vital']
unit_dist = unit_dist[cols]

display(unit_dist)

# Check for potential unit mismatches (e.g., Celsius values in Fahrenheit column)
print("\n--- Temperature Unit Mismatch Check ---")
print("Looking for values that might be in Celsius (32-42) instead of Fahrenheit (90-110)")

celsius_suspects = conn.execute("""
    SELECT value, COUNT(*) as count
    FROM silver_vitals
    WHERE loinc_code = '8310-5'
    AND value BETWEEN 32 AND 42
    GROUP BY value
    ORDER BY count DESC
    LIMIT 20
""").df()

if len(celsius_suspects) > 0:
    print(f"Found {celsius_suspects['count'].sum():,} temperature values in 32-42 range (possible Celsius):")
    display(celsius_suspects)
else:
    print("No temperature values in Celsius range detected.")

## Cell 10: Summary & Recommendations

In [ ]:
print("="*60)
print("SUMMARY & RECOMMENDATIONS")
print("="*60)

# Build summary table
summary_data = []
total_rows = conn.execute("SELECT COUNT(*) FROM silver_vitals").fetchone()[0]

for loinc, limits in LIMITS.items():
    stats = conn.execute(f"""
        SELECT
            COUNT(*) as total,
            SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) as nulls,
            SUM(CASE WHEN value = 0 THEN 1 ELSE 0 END) as zeros,
            SUM(CASE WHEN value < {limits['min']} THEN 1 ELSE 0 END) as below_min,
            SUM(CASE WHEN value > {limits['max']} THEN 1 ELSE 0 END) as above_max
        FROM silver_vitals
        WHERE loinc_code = '{loinc}'
    """).fetchone()
    
    total, nulls, zeros, below, above = stats
    issues = nulls + zeros + below + above
    
    summary_data.append({
        'vital': limits['name'],
        'loinc': loinc,
        'total_rows': total,
        'nulls': nulls,
        'zeros': zeros,
        'below_limit': below,
        'above_limit': above,
        'total_issues': issues,
        'issue_pct': round(100 * issues / total, 2) if total > 0 else 0
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== Data Quality Issues by Vital ===")
display(summary_df)

total_issues = summary_df['total_issues'].sum()
print(f"\nTotal rows: {total_rows:,}")
print(f"Total issues: {total_issues:,} ({100*total_issues/total_rows:.2f}%)")

print("\n" + "="*60)
print("PROPOSED CLEANING RULES")
print("="*60)

cleaning_rules = [
    ("1. Remove nulls", "Remove rows where value IS NULL"),
    ("2. Remove zeros", "Remove rows where value = 0 (not physiologically valid)"),
    ("3. Temperature", f"Remove where value < {LIMITS['8310-5']['min']} OR value > {LIMITS['8310-5']['max']}"),
    ("4. Heart rate", f"Remove where value < {LIMITS['8867-4']['min']} OR value > {LIMITS['8867-4']['max']}"),
    ("5. Respiratory rate", f"Remove where value < {LIMITS['9279-1']['min']} OR value > {LIMITS['9279-1']['max']}"),
    ("6. O2 saturation", f"Remove where value < {LIMITS['2708-6']['min']} OR value > {LIMITS['2708-6']['max']}"),
    ("7. Systolic BP", f"Remove where value < {LIMITS['8480-6']['min']} OR value > {LIMITS['8480-6']['max']}"),
    ("8. Diastolic BP", f"Remove where value < {LIMITS['8462-4']['min']} OR value > {LIMITS['8462-4']['max']}"),
]

for rule, desc in cleaning_rules:
    print(f"\n{rule}:")
    print(f"  {desc}")

print("\n" + "="*60)
print("ESTIMATED DATA RETENTION")
print("="*60)
retention_pct = 100 * (total_rows - total_issues) / total_rows
print(f"\nAfter cleaning: ~{total_rows - total_issues:,} rows ({retention_pct:.2f}% retention)")
print(f"Estimated data loss: ~{total_issues:,} rows ({100-retention_pct:.2f}%)")

In [ ]:
# Cleanup
conn.close()
print("Analysis complete. Connection closed.")